# Prose Interpreter Evaluation (Local)

This notebook is a dedicated local evaluation workspace for the prose-interpreter stress tests in `ChatbotLP`.

It is designed for local execution on a Mac in VS Code, Jupyter Lab, or Jupyter Notebook. It does **not** assume Google Colab and uses local imports from the repository.


## 1. Environment / Local Setup

This section configures repo-local imports, prints basic environment diagnostics, checks for `GEMINI_API_KEY`, and reports whether a supported solver appears to be available.


In [ ]:
import os

os.environ["GEMINI_API_KEY"] = ""
os.environ["LLM_PROVIDER"] = "gemini"
os.environ["GEMINI_MODEL"] = "gemini-3-flash-preview"

print("GEMINI_API_KEY available:", bool(os.getenv("GEMINI_API_KEY")))
print("LLM_PROVIDER:", os.getenv("LLM_PROVIDER"))
print("GEMINI_MODEL:", os.getenv("GEMINI_MODEL"))


In [ ]:
import os
import sys
import inspect
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("cwd:", Path.cwd())
print("repo root:", REPO_ROOT)
print("python:", sys.executable)
print("sys.path[0:5]:", sys.path[:5])

import src.llm_problem_interpreter as lpi
import src.chatbot_engine as ce

print("llm_problem_interpreter file:", inspect.getsourcefile(lpi))
print("chatbot_engine file:", inspect.getsourcefile(ce))
print("interpret_problem_from_text starts at:", inspect.getsourcelines(lpi.interpret_problem_from_text)[1])
print("run_chatbot_session starts at:", inspect.getsourcelines(ce.run_chatbot_session)[1])

print("\nSnippet from llm_problem_interpreter:")
print("".join(inspect.getsource(lpi.interpret_problem_from_text).splitlines(True)[:12]))

print("\nEnv:")
print("LLM_PROVIDER =", os.getenv("LLM_PROVIDER"))
print("GEMINI_API_KEY present =", bool(os.getenv("GEMINI_API_KEY")))


In [ ]:
from __future__ import annotations

import json
import os
import platform
import sys
from pathlib import Path
from pprint import pprint

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("Repo root:", REPO_ROOT)
print("Python executable:", sys.executable)
print("Python version:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Working directory:", Path.cwd())
print("GEMINI_API_KEY available:", bool(os.getenv("GEMINI_API_KEY")))
print("LLM_PROVIDER:", os.getenv("LLM_PROVIDER"))
print("GEMINI_MODEL:", os.getenv("GEMINI_MODEL"))


In [ ]:
from src.chatbot_engine import run_chatbot_session
from src.llm_problem_interpreter import summarize_problem_state
from src.prose_interpreter_evaluation import (
    build_benchmark_evaluation_cases,
    compare_problem_states,
    evaluate_benchmark_cases,
    summarize_failure_modes,
)
from src.schema import ProblemState
from src.solver import _get_solver
from src.validator import validate_state

solver_obj, solver_name, solver_path, solver_tried = _get_solver(
    solver_name="ipopt",
    fallback_solver="glpk",
    verbose=False,
)

print("Solver available:", solver_obj is not None)
print("Selected solver:", solver_name)
print("Solver executable:", solver_path)
print("Solver candidates tried:")
pprint(solver_tried)


## 2. Evaluation Utilities

These helpers keep notebook output readable while reusing the repo evaluation layer rather than duplicating parsing or scoring logic.


In [ ]:
CASES = {case["name"]: case for case in build_benchmark_evaluation_cases()}
print("Benchmark cases:", list(CASES.keys()))


def require_llm_configuration() -> None:
    if not os.getenv("GEMINI_API_KEY"):
        raise RuntimeError("GEMINI_API_KEY is not set in the current environment.")
    if os.getenv("LLM_PROVIDER", "").lower() != "gemini":
        raise RuntimeError("Set LLM_PROVIDER=gemini before running live prose evaluation.")


def evaluate_case(case_name: str, mode: str = "guided") -> dict:
    require_llm_configuration()
    case = CASES[case_name]
    session_result = run_chatbot_session(
        state=ProblemState(),
        user_message=case["prose"],
        mode=mode,
        use_llm=True,
    )
    interpreted_state = session_result.get("state", ProblemState())
    comparison = compare_problem_states(case["expected_state"], interpreted_state)
    return {
        "case_name": case_name,
        "label": case["label"],
        "prose_input": case["prose"],
        "semantic_plan": session_result.get("semantic_plan"),
        "problem_state": interpreted_state,
        "problem_state_summary": summarize_problem_state(interpreted_state),
        "validation_result": session_result.get("validation_result") or validate_state(interpreted_state),
        "solve_result": session_result.get("solve_result", {}),
        "solve_succeeded": bool(session_result.get("solve_result", {}).get("success", False)),
        "explanation_output": session_result.get("response", ""),
        "comparison": comparison,
        "error_categories": comparison.get("error_category_counts", {}),
    }


def display_case_result(result: dict) -> None:
    print("Label:\n")
    print(result["label"])

    print("\nProse input:\n")
    print(result["prose_input"])

    print("\nSemantic plan:\n")
    print(json.dumps(result["semantic_plan"], indent=2, default=str))

    print("\nBuilt ProblemState:\n")
    print(json.dumps(result["problem_state"].to_dict(), indent=2, default=str))

    print("\nProblemState summary:\n")
    print(json.dumps(result["problem_state_summary"], indent=2, default=str))

    print("\nValidation output:\n")
    print(json.dumps(result["validation_result"], indent=2, default=str))

    print("\nSolve status/result:\n")
    print(json.dumps(result["solve_result"], indent=2, default=str))
    print("Solve succeeded:", result["solve_succeeded"])

    print("\nExplanation output:\n")
    print(result["explanation_output"])

    print("\nScoring vs expected benchmark state:\n")
    print(json.dumps(result["comparison"], indent=2, default=str))

    print("\nError categories:\n")
    print(json.dumps(result["error_categories"], indent=2, default=str))


## 3. Benchmark Prose Cases

The evaluation set includes:

- Case A canonical prose
- Case A paraphrased prose
- Case A incomplete prose
- Case A ambiguous prose
- Case B negative-bid prose
- Case C transformation prose


In [ ]:
for case_name, case in CASES.items():
    print(f"{case_name}: {case['label']}")
    print(case["prose"])
    print("-" * 80)


## 4. Per-Case Evaluation Cells

Run each cell independently to inspect the full prose-to-solve pipeline for that benchmark-style input.


### Case A Canonical Prose

In [ ]:
result_case_a_canonical = evaluate_case("canonical_case_a")
display_case_result(result_case_a_canonical)


### Case A Paraphrased Prose

In [ ]:
result_case_a_paraphrased = evaluate_case("paraphrased_case_a")
display_case_result(result_case_a_paraphrased)


### Case A Incomplete Prose

In [ ]:
result_case_a_incomplete = evaluate_case("incomplete_case_a")
display_case_result(result_case_a_incomplete)


### Case A Ambiguous Prose

In [ ]:
result_case_a_ambiguous = evaluate_case("ambiguous_case_a")
display_case_result(result_case_a_ambiguous)


### Case B Negative-Bid Prose

In [ ]:
result_case_b_negative_bid = evaluate_case("negative_bid_case_b")
display_case_result(result_case_b_negative_bid)


### Case C Transformation Prose

In [ ]:
result_case_c_transformation = evaluate_case("transformation_case_c")
display_case_result(result_case_c_transformation)


## 5. Aggregate Comparison Section

This section runs all benchmark cases, then summarizes field-level mismatches and dominant error categories across the suite.


In [ ]:
require_llm_configuration()
aggregate_report = evaluate_benchmark_cases(mode="guided")
aggregate_results = aggregate_report["cases"]
aggregate_failure_modes = summarize_failure_modes(aggregate_results)

aggregate_summary = []
for case in aggregate_results:
    comparison = case["comparison"]
    aggregate_summary.append(
        {
            "label": case["label"],
            "solve_succeeded": case["solve_succeeded"],
            "passed_fields": len(comparison["passed_fields"]),
            "failed_fields": len(comparison["failed_fields"]),
            "missing_fields": len(comparison["missing_fields"]),
            "extra_fields": len(comparison["extra_fields"]),
            "error_categories": comparison["error_category_counts"],
        }
    )

print("Aggregate per-case summary:\n")
print(json.dumps(aggregate_summary, indent=2, default=str))

print("\nDominant error categories across cases:\n")
print(json.dumps(aggregate_failure_modes, indent=2, default=str))

print("\nNotebook-readable summary:\n")
for row in aggregate_summary:
    print(
        f"- {row['label']}: solve_succeeded={row['solve_succeeded']}, "
        f"passed={row['passed_fields']}, failed={row['failed_fields']}, "
        f"missing={row['missing_fields']}, extra={row['extra_fields']}"
    )


## 6. Notes / Interpretation

When reviewing results, pay special attention to these failure types:

- `wrong_numeric_value`: the model captured the right entity but changed a price, quantity, capacity, or yield coefficient.
- `missing_entity`: an expected supplier, consumer, node, product, transport link, technology, or bid was omitted.
- `extra_entity`: the interpreter invented extra structure not clearly supported by the prose.
- `wrong_owner_relation`: a bid was attached to the wrong supplier, consumer, transport link, or technology.
- `wrong_product_mapping`: an entity or bid was attached to the wrong product.
- `wrong_node_mapping`: an entity or transport endpoint was attached to the wrong node.
- `missing_capacity`, `missing_price`, `missing_quantity`: the interpreter preserved the entity but left key solver-relevant values unspecified.

For Case A, the most important questions are usually whether the interpreter preserved the numerical values and did not over-invent structure. For Case B and Case C, also check whether negative bids and transformation yields were captured faithfully rather than simplified back into a plain Case A shape.
